In [0]:
import unittest

class TestPlayerPipeline(unittest.TestCase):
    
    @classmethod
    def setUpClass(cls):
        cls.spark = spark

    def test_load_data(self):
        test_uc_table = "test_table"
        mock_data = [
            ("0029600034", 184, "Bobby", "Phills", 14, "1610612739", "Cleveland", "Cavaliers", "CLE"),
            # (2, "Klay Thompson", "GSW", False),
            # (3, "LeBron James", "LAL", True)
        ]
        mock_schema = ["game_id", "player_id", "first_name", "last_name", "jersey_num", "team_id", "team_city", "team_name", "team_abbreviation"]
        
        df_mock = spark.createDataFrame(mock_data, schema=mock_schema)
        df_mock.createOrReplaceTempView(test_uc_table)

        result_df = load_data(test_uc_table)
        actual_rows = result_df.collect()
        
        self.assertEqual(result_df.count(), 1)
        
        self.assertEqual(actual_rows[0]["game_id"], "0029600034")
        self.assertEqual(actual_rows[0]["team_city"], "Cleveland")
        self.assertEqual(actual_rows[0]["team_abbreviation"], "CLE")

    def test_get_star_players(self):
        # "player_id", "first_name", "last_name", "display_first_last", "position", "from_year", "to_year"
        common_player_info = [
            (10, "Kareem", "Abdul-Jabbar", "Kareem Abdul-Jabbar", "Center", 1969, 1988, "Y"),
            (119, "John", "Abramovic", "John Abramovic", "Forward", 1970, 1987, "N"),
            (217, "Stephen", "Curry",  "Stephen Curry", "Guard", 2009, 2022, "Y"),
            # (76003, "Kareem", "Abdul-Jabbar", "Y")
            # (76003, "Kareem", "Abdul-Jabbar", "Y")
        ]
        common_player_info_schema = ["person_id", "first_name", "last_name", "display_first_last", "position", "from_year", "to_year", "greatest_75_flag"]
        common_player_info_df = spark.createDataFrame(common_player_info, schema=common_player_info_schema)

        inactive_players = [
            ("0029600034", 10, "first1", "last1"),
            ("0029600068", 10, "first1", "last1"),
            ("0029600070", 10, "first1", "last1"),
            ("0029600071", 10, "first1", "last1"),
            ("0029600072", 10, "first1", "last1"),
            ("0029600073", 10, "first1", "last1"),
            ("0029600074", 10, "first1", "last1"),
            ("0029600075", 10, "first1", "last1"),
            ("0029600076", 10, "first1", "last1"),
            ("0029600077", 10, "first1", "last1"),
            ("0029600078", 10, "first1", "last1"),
            ("0029900819", 119, "first2", "last2"),
            ("0029900836", 119, "first2", "last2"),
            ("0029900855", 119, "first2", "last2"),
            ("0020201123", 217, "first3", "last3"),
            ("0020201145", 217, "first3", "last3"),
            ("0020201155", 217, "first3", "last3"),
            ("0020201231", 321, "first3", "last3"),
            ("0020201356", 321, "first3", "last3"),
            ("0020201566", 321, "first3", "last3"),
        ]
        inactive_players_schema = ["game_id", "player_id", "first_name", "last_name"]
        inactive_players_df = spark.createDataFrame(inactive_players, schema=inactive_players_schema)

        result_df = get_star_players(common_player_info_df, inactive_players_df)
        actual_rows = result_df.collect()
        
        self.assertEqual(len(actual_rows), 1)
        self.assertEqual(actual_rows[0]["player_id"], 10)
        self.assertEqual(actual_rows[0]["first_name"], "Kareem")
        self.assertEqual(actual_rows[0]["last_name"], "Abdul-Jabbar")

    def test_process_game_with_star_absence(self):
        inactive_players = [
            ("0029600034", 10, "first1", "last1"),
            ("0029600068", 10, "first1", "last1"),
            ("0029600070", 10, "first1", "last1"),
            ("0029900819", 119, "first2", "last2"),
            ("0029900836", 119, "first2", "last2"),
            ("0029900855", 119, "first2", "last2"),
            ("0020201123", 217, "first3", "last3"),
            ("0020201145", 217, "first3", "last3"),
            ("0020201155", 217, "first3", "last3"),
            ("0020201231", 321, "first3", "last3"),
            ("0020201356", 321, "first3", "last3"),
            ("0020201566", 321, "first3", "last3"),
        ]

        inactive_players_schema = ["game_id", "player_id", "first_name", "last_name"]
        inactive_players_df = spark.createDataFrame(inactive_players, schema=inactive_players_schema)

        star_players = [
            (10, "first1", "last1"),
            (119, "first2", "last2"),
            (217, "first3", "last3"),
        ]
        star_players_schema = ["player_id", "first_name", "last_name"]
        star_players_df = spark.createDataFrame(star_players, schema=star_players_schema)

        game_with_star_absence_df = process_game_with_star_absence(inactive_players_df, star_players_df)
        display(game_with_star_absence_df)
        actual_rows = game_with_star_absence_df.collect()
        
        self.assertEqual(len(actual_rows), 9)

    def test_process_star_absence_games(self):
        game_with_star_absence = [
            ("0029600034", "1610610035", 10, "first1", "last1"),
            ("0029600068", "1610610035", 10, "first1", "last1"),
            ("0029600070", "1610610035", 10, "first1", "last1"),
            ("0029900819", "1610610021", 119, "first2", "last2"),
            ("0029900836", "1610612712", 119, "first2", "last2"),
            ("0029900855", "1610612713", 119, "first2", "last2"),
            ("0020201123", "1610612755", 217, "first3", "last3"),
            ("0020201145", "1610612756", 217, "first3", "last3"),
            ("0020201155", "1610612756", 217, "first3", "last3"),
        ]

        game_with_star_absence_schema = ["game_id", "team_id", "player_id", "first_name", "last_name"]
        game_with_star_absence_df = spark.createDataFrame(game_with_star_absence, schema=game_with_star_absence_schema)

        game = [
            (21946, "0029600034", 10, "1946-11-01 00:00:00", "1610610035", "W", "1610612752", "L", "Regular Season"),
            (21946, "0029600068", 10, "1946-11-01 00:00:00", "1610610035", "W", "1610612752", "L", "Regular Season"),
            (21946, "0029600070", 10, "1946-11-01 00:00:00", "1610610035", "W", "1610612752", "L", "Playoffs"),
            (21946, "0029900819", 119, "1946-11-01 00:00:00", "1610610021", "W", "1610612767", "L", "Regular Season"),
            (21946, "0029900836", 119, "1946-11-01 00:00:00", "1610612712", "W", "1610612767", "L", "Regular Season"),
            (21946, "0029900855", 119, "1946-11-01 00:00:00", "1610612713", "W", "1610612767", "L", "Regular Season"),
            (21946, "0020201123", 217, "1946-11-01 00:00:00", "1610610098", "W", "1610612755", "L", "Regular Season"),
            (21946, "0020201145", 217, "1946-11-01 00:00:00", "1610610098", "W", "1610612756", "L", "Regular Season"),
            (21946, "0020201155", 217, "1946-11-01 00:00:00", "1610610098", "W", "1610612756", "L", "Regular Season"),
        ]
        game_schema = ["season_id", "game_id", "player_id", "game_date", "team_id_home", "wl_home", "team_id_away", "wl_away", "season_type"]
        game_df = spark.createDataFrame(game, schema=game_schema)

        game_with_winning_and_losing_df = process_star_absence_games(game_with_star_absence_df, game_df)
        display(game_with_winning_and_losing_df)
        actual_rows = game_with_winning_and_losing_df.collect()

        self.assertEqual(len(actual_rows), 8)
        self.assertEqual(actual_rows[0]["game_id"], "0020201123")
        self.assertEqual(actual_rows[0]["team_id_home"], "1610610098")
        self.assertEqual(actual_rows[0]["team_id_away"], "1610612755")
        self.assertEqual(actual_rows[0]["team_id"], "1610612755")
        self.assertEqual(actual_rows[1]["game_id"], "0020201145")
        self.assertEqual(actual_rows[1]["team_id_home"], "1610610098")
        self.assertEqual(actual_rows[1]["team_id_away"], "1610612756")
        self.assertEqual(actual_rows[1]["team_id"], "1610612756")
        self.assertEqual(actual_rows[2]["game_id"], "0020201155")
        self.assertEqual(actual_rows[2]["team_id_home"], "1610610098")
        self.assertEqual(actual_rows[2]["team_id_away"], "1610612756")
        self.assertEqual(actual_rows[2]["team_id"], "1610612756")
        self.assertEqual(actual_rows[3]["game_id"], "0029600034")
        self.assertEqual(actual_rows[3]["team_id_home"], "1610610035")
        self.assertEqual(actual_rows[3]["team_id_away"], "1610612752")
        self.assertEqual(actual_rows[3]["team_id"], "1610610035")
        self.assertEqual(actual_rows[4]["game_id"], "0029600068")
        self.assertEqual(actual_rows[4]["team_id_home"], "1610610035")
        self.assertEqual(actual_rows[4]["team_id_away"], "1610612752")
        self.assertEqual(actual_rows[4]["team_id"], "1610610035")
        self.assertEqual(actual_rows[5]["game_id"], "0029900819")
        self.assertEqual(actual_rows[5]["team_id_home"], "1610610021")
        self.assertEqual(actual_rows[5]["team_id_away"], "1610612767")
        self.assertEqual(actual_rows[5]["team_id"], "1610610021")
        self.assertEqual(actual_rows[6]["game_id"], "0029900836")
        self.assertEqual(actual_rows[6]["team_id_home"], "1610612712")
        self.assertEqual(actual_rows[6]["team_id_away"], "1610612767")
        self.assertEqual(actual_rows[6]["team_id"], "1610612712")
        self.assertEqual(actual_rows[7]["game_id"], "0029900855")
        self.assertEqual(actual_rows[7]["team_id_home"], "1610612713")
        self.assertEqual(actual_rows[7]["team_id_away"], "1610612767")
        self.assertEqual(actual_rows[7]["team_id"], "1610612713")